In [2]:
import os
import torch
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionXLImg2ImgPipeline,
    StableDiffusionInstructPix2PixPipeline,
    DPMSolverMultistepScheduler
)
import zipfile
from PIL import Image

#### Stable Diffusion

In [4]:
device = "cuda"
model_id: str = "runwayml/stable-diffusion-v1-5",
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

def generate_image(
    prompt: str,
    output_path: str = "generated.png",
    height: int = 512,
    width: int = 512,
    num_inference_steps: int = 50,
    guidance_scale: float = 7.5,
):
       
    image = pipe(
        prompt,
        height=height,
        width=width,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale
    ).images[0]

    image.save(output_path)
    print(f"Image saved to {output_path}")


In [ ]:
import json
apac_file_path : str = "../data/synthetic_data_generation/apac_gender_neutral_prompts.json"

with open(apac_file_path, "r") as f:
    data = json.load(f)
    data = data["data"]
    
type(data)

In [ ]:
output_folder : str = "../data/stable-diffusion/APAC"

for idx, desc in enumerate(data):
    generate_image(
        prompt=desc,
        output_path=f"{output_folder}/{idx}.png"
    )

#### Stable Diffusion XL

In [1]:
def gen_sdxl(prompt, output_path):
    pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16,
    ).to("cuda")
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    init_image = Image.new("RGB", (1024, 1024), color="white")
    img = pipe(
        prompt=prompt,
        image=init_image,
        strength=0.75,
        num_inference_steps=50,
        guidance_scale=2.0
    ).images[0]
    img.save(output_path)
    print(f"Image saved to {output_path}")

In [ ]:
output_folder : str = "../data/stable-diffusion-XL/APAC"

for idx, desc in enumerate(data):
    gen_sdxl(
        prompt=desc,
        output_path=f"{output_folder}/{idx}.png"
    )


In [ ]:
def zip_folder(folder_path: str, zip_path: str):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(folder_path):
            for fname in files:
                full = os.path.join(root, fname)
                zipf.write(full, arcname=fname)
    print(f"Created ZIP archive at {zip_path}")